**MGMT298D: Science and Strategy of AI**
# Week 4: Neural Networks

#### This notebook introduces neural networks for classification using `keras` (via `tensorflow`). We build progressively deeper networks on the MNIST handwritten-digit dataset and use EarlyStopping to avoid overfitting.

# 1 Setup & Data

#### We import `numpy`, `sklearn.metrics`, and `tensorflow.keras`, load MNIST (28×28 grayscale images of digits 0–9), normalize pixel values to [0, 1], and flatten each image to a 784-dim vector.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score
from tensorflow.keras import Sequential
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

rng = np.random.default_rng(42)
train_idx = rng.choice(len(X_train), 2000, replace=False)
test_idx  = rng.choice(len(X_test),   500, replace=False)
X_train, y_train = X_train[train_idx], y_train[train_idx]
X_test,  y_test  = X_test[test_idx],   y_test[test_idx]

fig, axes = plt.subplots(2, 4, figsize=(6, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

X_train = X_train.reshape(-1, 784) / 255.0
X_test  = X_test.reshape(-1, 784) / 255.0

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Features: {X_train.shape[1]}")

---
# 2 Simple Neural Network

#### We start with the simplest possible network — one hidden layer with 4 neurons and relu activation, followed by a 10-unit softmax output for 10-class digit classification.

In [ ]:
model_simple = Sequential([
    Dense(4, activation='relu', input_shape=(784,)),
    Dense(10, activation='softmax')
])

model_simple.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

history_simple = model_simple.fit(X_train, y_train, epochs=50,
                                   validation_split=0.15)

y_pred_simple = np.argmax(model_simple.predict(X_test, verbose=0), axis=1)
acc_simple = accuracy_score(y_test, y_pred_simple)
print(f"Simple NN Test Accuracy: {acc_simple:.4f}")

In [ ]:
plt.plot(history_simple.history['loss'], label='Train')
plt.plot(history_simple.history['val_loss'], label='Val')
plt.xlim(0, history_simple.params['epochs'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Simple NN Training')
plt.legend()
plt.show()

---
# 3 Deep Neural Network

#### Adding more layers (32 → 64 → 32) lets the network learn richer representations of digit structure, but without regularization it may overfit.

In [ ]:
model_deep = Sequential([
    Dense(32, activation='relu', input_shape=(784,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

model_deep.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

history_deep = model_deep.fit(X_train, y_train, epochs=50,
                               validation_split=0.15)

y_pred_deep = np.argmax(model_deep.predict(X_test, verbose=0), axis=1)
acc_deep = accuracy_score(y_test, y_pred_deep)
print(f"Deep NN Test Accuracy: {acc_deep:.4f}")

In [ ]:
plt.plot(history_deep.history['loss'], label='Train')
plt.plot(history_deep.history['val_loss'], label='Val')
plt.xlim(0, history_deep.params['epochs'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Deep NN Training')
plt.legend()
plt.show()

---
# 4 Regularized Neural Network

#### We train the same deep architecture but add `EarlyStopping`, which halts training when validation loss stops improving and restores the best weights. This is a simple, effective way to prevent overfitting.

In [ ]:
model_reg = Sequential([
    Dense(32, activation='relu', input_shape=(784,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

model_reg.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

early_stop = EarlyStopping(patience=5, restore_best_weights=True)

history_reg = model_reg.fit(X_train, y_train, epochs=50,
                             validation_split=0.15, callbacks=[early_stop])

y_pred_reg = np.argmax(model_reg.predict(X_test, verbose=0), axis=1)
acc_reg = accuracy_score(y_test, y_pred_reg)
print(f"Regularized NN Test Accuracy: {acc_reg:.4f}")

print("\nConfusion matrix (rows=true, cols=predicted):")
print(confusion_matrix(y_test, y_pred_reg))

In [ ]:
plt.plot(history_reg.history['loss'], label='Train')
plt.plot(history_reg.history['val_loss'], label='Val')
plt.xlim(0, history_reg.params['epochs'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Regularized NN Training')
plt.legend()
plt.show()

---
# 5 Model Comparison

#### Final side-by-side comparison of all three models on test accuracy.

In [ ]:
print(f"Simple NN:       {acc_simple:.4f}")
print(f"Deep NN:         {acc_deep:.4f}")
print(f"Regularized NN:  {acc_reg:.4f}")

In [ ]:
models = ['Simple NN', 'Deep NN', 'Regularized NN']
accs = [acc_simple, acc_deep, acc_reg]
plt.bar(models, accs)
plt.ylabel('Test Accuracy')
plt.title('Model Comparison')
plt.ylim(0, 1)
plt.show()

---
# 6 Sample Predictions

#### A look at what the regularized model gets right and wrong on the test set — four correctly classified digits on top, four misclassifications on the bottom.

In [ ]:
correct_idx = np.where(y_pred_reg == y_test)[0][:4]
wrong_idx   = np.where(y_pred_reg \!= y_test)[0][:4]

fig, axes = plt.subplots(2, 4, figsize=(8, 4))
for col, idx in enumerate(correct_idx):
    ax = axes[0, col]
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"True: {y_test[idx]}  Pred: {y_pred_reg[idx]}")
    ax.axis('off')
for col, idx in enumerate(wrong_idx):
    ax = axes[1, col]
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"True: {y_test[idx]}  Pred: {y_pred_reg[idx]}")
    ax.axis('off')
plt.tight_layout()
plt.show()